# Notebook 1 — Fine-tune FinBERT for Financial Sentiment (Pipeline 1)
**ISOM5240 Group Project — Earnings Call Dividend Risk Early Warning System**

This notebook fine-tunes `ProsusAI/finbert` on the Financial PhraseBank dataset
for 3-class sentiment classification: `positive`, `negative`, `neutral`.

The fine-tuned model is pushed to HuggingFace Hub and used as Pipeline 1
in the Streamlit app to score each transcript chunk for financial negativity.

**Steps:**
1. Install packages
2. Load & explore Financial PhraseBank dataset
3. Tokenize and preprocess
4. Fine-tune FinBERT with HuggingFace Trainer
5. Evaluate accuracy and F1
6. Test inference on sample sentences
7. Push fine-tuned model to HuggingFace Hub


In [ ]:
# ── Step 1: Install required packages ──────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn huggingface_hub
!pip install -q sentencepiece

In [ ]:
# ── Step 2: Imports ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
print('PyTorch version:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Step 3: HuggingFace login (enter your token when prompted) ──────────────
from huggingface_hub import notebook_login
notebook_login()

# Set your HuggingFace username here
HF_USERNAME = 'your-username'            # ← CHANGE THIS
MODEL_NAME   = 'finbert-dividend-sentiment'
FULL_MODEL_ID = f'{HF_USERNAME}/{MODEL_NAME}'

In [ ]:
# ── Step 4: Load Financial PhraseBank dataset ───────────────────────────────
# Using 'sentences_allagree' split: highest label agreement (most reliable)
raw = load_dataset('financial_phrasebank', 'sentences_allagree')
print(raw)
print('\nLabel mapping:', raw['train'].features['label'])
print('\nSample entries:')
for i in range(3):
    print(f'  [{raw["train"][i]["label"]}] {raw["train"][i]["sentence"]}')

In [ ]:
# ── Step 5: Explore class distribution ─────────────────────────────────────
from collections import Counter
labels = raw['train']['label']
dist = Counter(labels)
label_names = raw['train'].features['label'].names   # ['negative','neutral','positive']
print('Class distribution:')
for k, v in sorted(dist.items()):
    print(f'  {label_names[k]:10s}: {v}')
print(f'\nTotal samples: {len(labels)}')

In [ ]:
# ── Step 6: Train / Validation / Test split ─────────────────────────────────
# Financial PhraseBank only has a 'train' split — we create val and test splits
split1 = raw['train'].train_test_split(test_size=0.2, seed=42, stratify_by_column='label')
split2 = split1['test'].train_test_split(test_size=0.5, seed=42, stratify_by_column='label')

dataset = DatasetDict({
    'train': split1['train'],
    'validation': split2['train'],
    'test': split2['test'],
})
print('Split sizes:')
for split, ds in dataset.items():
    print(f'  {split:12s}: {len(ds)} samples')

In [ ]:
# ── Step 7: Tokenize ────────────────────────────────────────────────────────
BASE_MODEL = 'ProsusAI/finbert'
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_fn(examples):
    return tokenizer(
        examples['sentence'],
        truncation=True,
        padding='max_length',
        max_length=128,     # FinPhraseBank sentences are short
    )

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids','attention_mask','labels'])
print('Tokenization complete.')
print('Sample input_ids shape:', tokenized['train'][0]['input_ids'].shape)

In [ ]:
# ── Step 8: Load pre-trained FinBERT and replace head ───────────────────────
NUM_LABELS = 3   # negative=0, neutral=1, positive=2

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label={0:'negative', 1:'neutral', 2:'positive'},
    label2id={'negative':0, 'neutral':1, 'positive':2},
    ignore_mismatched_sizes=True,
)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:    {total_params:,}')
print(f'Trainable params:{trainable:,}')

In [ ]:
# ── Step 9: Define evaluation metrics ──────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

In [ ]:
# ── Step 10: Training arguments ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = f'./{MODEL_NAME}',
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    push_to_hub                 = True,
    hub_model_id                = FULL_MODEL_ID,
)
print('Training arguments set.')

In [ ]:
# ── Step 11: Instantiate Trainer and train ──────────────────────────────────
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized['train'],
    eval_dataset    = tokenized['validation'],
    tokenizer       = tokenizer,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Starting training ...')
train_result = trainer.train()
print('\nTraining complete!')
print(f'Training runtime: {train_result.metrics["train_runtime"]:.1f} s')
print(f'Samples/second:   {train_result.metrics["train_samples_per_second"]:.1f}')

In [ ]:
# ── Step 12: Evaluate on test set ───────────────────────────────────────────
test_results = trainer.evaluate(tokenized['test'])
print('\n── Test Set Results ──────────────────────────────')
for k, v in test_results.items():
    print(f'  {k:30s}: {v:.4f}')

# Detailed classification report
preds_out  = trainer.predict(tokenized['test'])
y_pred     = np.argmax(preds_out.predictions, axis=-1)
y_true     = preds_out.label_ids
print('\n── Classification Report ─────────────────────────')
print(classification_report(y_true, y_pred, target_names=['negative','neutral','positive']))

In [ ]:
# ── Step 13: Test inference on sample sentences ─────────────────────────────
sample_sentences = [
    'The company reported record-breaking revenue growth and raised its dividend by 10%.',
    'Management is reviewing the sustainability of the current payout ratio.',
    'Free cash flow turned negative; the company may need to suspend its dividend.',
    'We expect stable earnings in line with analyst consensus estimates.',
    'Debt covenants were breached; lenders have issued a waiver pending restructuring.',
]

sent_pipeline = pipeline(
    'text-classification', model=trainer.model,
    tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1,
    top_k=None,
)

print('Inference results:')
for sent in sample_sentences:
    res = sent_pipeline(sent[:512])
    scores = {item['label']: f"{item['score']:.3f}" for item in res[0]}
    print(f'  [{max(res[0], key=lambda x: x["score"])["label"].upper():8s}] {sent[:60]}...')
    print(f'           Scores: {scores}')

In [ ]:
# ── Step 14: Save and push model to HuggingFace Hub ─────────────────────────
trainer.save_model(f'./{MODEL_NAME}')
tokenizer.save_pretrained(f'./{MODEL_NAME}')
trainer.push_to_hub(commit_message='Fine-tuned FinBERT for dividend sentiment analysis')
print(f'\n✅ Model pushed to: https://huggingface.co/{FULL_MODEL_ID}')
print('   Use this ID in app.py → SENTIMENT_MODEL_ID')